# 09 — How far behind the betting market?

The betting market is the hardest benchmark there is: thousands of people with money on
the line, pricing in everything public, including things our data can't see (injuries,
weight cuts, camp news). Until now its log loss was an *estimate*. This notebook measures
it, on the same fights as our model.

**The odds** (`data/odds.py`, Kaggle `jerzyszocik/ufc-betting-odds-daily-dataset`). Every
row carries the ufcstats fight URL, so odds match our fights exactly by id. For 2010 to
mid-2025 it holds one line per fight from bestfightodds.com (via the mdabbert dataset).
Those lines best match the bookmakers' consensus from the final day before a fight:
**near-closing lines**, the standard benchmark.

**From odds to a probability:** decimal odds 1.50 / 2.50 give raw probabilities 0.667 and
0.400. They add up to 1.067; the extra 6.7% is the bookmaker's margin. Dividing each by
1.067 removes it: **62.5%** and **37.5%**.

**Two caveats, stated up front:**

- **Coverage.** The source has no odds for 18% of 2023 fights and 25% of 2024 fights.
  Every comparison below uses only the fights that do have odds, so model and market are
  always scored on exactly the same fights.
- **The test years stay closed.** The market is scored on 2014–2024 only; its 2025–26
  score waits for the final test look, side by side with ours.

In [1]:
import json

import numpy as np
import pandas as pd

from ufc_predictor.config import RAW_DIR
from ufc_predictor.data.io import load_processed
from ufc_predictor.data.odds import load_market, market_for
from ufc_predictor.features import manifest
from ufc_predictor.modeling.calibrate import load_calibration, stretch
from ufc_predictor.modeling.evaluate import compare, compare_log_loss, is_womens_fight, score
from ufc_predictor.modeling.model import fit_models, predict_proba_mean
from ufc_predictor.modeling.split import chronological_split
from ufc_predictor.modeling.tune import cv_predictions, load_params, seed_columns

odds_source = json.loads((RAW_DIR / "odds" / "ODDS.json").read_text())
print(f"{odds_source['dataset']} version {odds_source['version']} (downloaded {odds_source['downloaded_on']})")

market = load_market()
featured = load_processed("fights_featured")
train, validation, _ = chronological_split(featured)      # the test years stay closed
params = load_params()
a = load_calibration()

train = train.assign(p_market=market_for(train, market).to_numpy())
validation = validation.assign(p_market=market_for(validation, market).to_numpy())

# our final predictor on the validation years: tuned settings, 8 seeds, calibrated
X_train, y_train = manifest.make_xy(train)
X_val, _ = manifest.make_xy(validation)
validation["p_model_raw"] = predict_proba_mean(fit_models(X_train, y_train, params), X_val)
validation["p_model"] = stretch(validation["p_model_raw"].to_numpy(), a)
validation["mens"] = ~is_womens_fight(validation, history=featured).to_numpy()
print(f"calibration stretch a = {a:.3f}")

jerzyszocik/ufc-betting-odds-daily-dataset version 259 (downloaded 2026-10-05)


calibration stretch a = 1.054


## 1. Coverage: which fights have odds

In [2]:
both = pd.concat([train.assign(period="train"), validation.assign(period="validation")], ignore_index=True)
both["year"] = both["event_date"].dt.year
coverage = both.groupby(["period", "year"]).agg(
    fights=("fights_url", "size"), with_odds=("p_market", lambda p: p.notna().sum()))
coverage["share"] = coverage["with_odds"] / coverage["fights"]
coverage.round(3)

fights  with_odds  share
period     year                          
train      2014     494        469  0.949
           2015     464        439  0.946
           2016     483        463  0.959
           2017     446        418  0.937
           2018     469        451  0.962
           2019     506        482  0.953
           2020     444        421  0.948
           2021     497        478  0.962
           2022     506        480  0.949
validation 2023     504        416  0.825
           2024     513        370  0.721

## 2. The market's own track record

Accuracy here is how often the betting favourite won. Year to year, the market moves
around more than you might expect: some years simply have more upsets.

In [3]:
rows = []
for (period, year), g in both[both["p_market"].notna()].groupby(["period", "year"]):
    s = score(g["fighter_a_won"], g["p_market"])
    rows.append({"period": period, "year": year, "fights": s["fights"],
                 "favourite_won": s["accuracy"], "log_loss": s["log_loss"], "auc": s["auc"]})
pd.DataFrame(rows).set_index(["period", "year"]).round(4)

fights  favourite_won  log_loss     auc
period     year                                         
train      2014     469         0.6802    0.6014  0.7385
           2015     439         0.6105    0.6295  0.6922
           2016     463         0.6242    0.6261  0.7007
           2017     418         0.6770    0.6123  0.7255
           2018     451         0.6419    0.6014  0.7361
           2019     482         0.6390    0.6282  0.6969
           2020     421         0.6615    0.6154  0.7220
           2021     478         0.6391    0.6294  0.6946
           2022     480         0.6760    0.6036  0.7352
validation 2023     416         0.6959    0.5864  0.7616
           2024     370         0.7000    0.5910  0.7550

## 3. Head to head: validation years, men's fights

Every row is scored on the same fights: men's validation fights that have odds. The
**50/50 blend** averages the model's and the market's probabilities. If the blend beats
the market, the model knows something the market doesn't.

In [4]:
h = validation[validation["mens"] & validation["p_market"].notna()]
y = h["fighter_a_won"].to_numpy()
blend = (h["p_model"] + h["p_market"]) / 2
head = pd.DataFrame({
    name: score(y, p) for name, p in [
        ("coin flip", np.full(len(y), 0.5)),
        ("our model, uncalibrated", h["p_model_raw"]),
        ("our model", h["p_model"]),
        ("betting market", h["p_market"]),
        ("50/50 blend", blend),
    ]
}).T
head.round(4)

,fights,accuracy,log_loss,auc
coin flip,664.0,0.5000,0.6931,0.5000
"our model, uncalibrated",664.0,0.6235,0.6438,0.6860
our model,664.0,0.6235,0.6428,0.6860
betting market,664.0,0.6935,0.5847,0.7634
50/50 blend,664.0,0.7048,0.6041,0.7553


In [5]:
gap = compare_log_loss(y, h["p_model"], h["p_market"])
picks = compare(y, h["p_model"], h["p_market"])
blend_vs_market = compare_log_loss(y, blend, h["p_market"])
covered = (head.loc["coin flip", "log_loss"] - head.loc["our model", "log_loss"]) / (
    head.loc["coin flip", "log_loss"] - head.loc["betting market", "log_loss"])

print(f"model minus market, log loss: {gap['difference']:+.4f} (standard error {gap['se']:.4f}, p = {gap['p_value']:.4f})")
print(f"the model covers {covered:.0%} of the distance from a coin flip to the market")
disagree = picks["fixes"] + picks["breaks"]
print(f"they pick different winners in {disagree} of {len(y)} fights: "
      f"the model was right in {picks['fixes']}, the market in {picks['breaks']} (p = {picks['p_value']:.3f})")
print(f"blend minus market, log loss: {blend_vs_market['difference']:+.4f} (p = {blend_vs_market['p_value']:.3f})")

model minus market, log loss: +0.0581 (standard error 0.0106, p = 0.0000)
the model covers 46% of the distance from a coin flip to the market
they pick different winners in 173 of 664 fights: the model was right in 64, the market in 109 (p = 0.001)
blend minus market, log loss: +0.0194 (p = 0.000)


Are the fights without odds different? If our model found them much easier or harder
than the fights with odds, the comparison above would not speak for the whole period.

In [6]:
men = validation[validation["mens"]]
rows = {}
for label, g in [("with odds", men[men["p_market"].notna()]), ("without odds", men[men["p_market"].isna()])]:
    debut = (g["fighter_a_prior_is_debut"] | g["fighter_b_prior_is_debut"]).mean()
    rows[label] = score(g["fighter_a_won"], g["p_model"]) | {"share_with_a_debutant": debut}
pd.DataFrame(rows).T.round(4)

,fights,accuracy,log_loss,auc,share_with_a_debutant
with odds,664.0,0.6235,0.6428,0.6860,0.1943
without odds,173.0,0.6127,0.6746,0.6159,0.2197


## 4. A second, bigger comparison: 2019–2022, out of fold

The same question on the training years, using the cross-validation predictions: each year
predicted by models trained only on the years before it, so none of them saw the fights
they predict. These models had less to learn from than the final one (2014–2018 for 2019,
and so on), and they are uncalibrated, so if anything they understate the model.

In [7]:
cv = cv_predictions(train, params)
cv["p_model"] = cv[seed_columns(cv)].mean(axis=1)
cv = cv.merge(train[["fights_url", "p_market"]], on="fights_url")
cv["mens"] = ~is_womens_fight(train.set_index("fights_url").loc[cv["fights_url"]].reset_index(),
                              history=featured).to_numpy()
c = cv[cv["mens"] & cv["p_market"].notna()]

rows = []
for year, g in list(c.groupby("year")) + [("2019-2022", c)]:
    model, mkt = score(g["fighter_a_won"], g["p_model"]), score(g["fighter_a_won"], g["p_market"])
    rows.append({"year": year, "fights": model["fights"],
                 "model_accuracy": model["accuracy"], "market_accuracy": mkt["accuracy"],
                 "model_log_loss": model["log_loss"], "market_log_loss": mkt["log_loss"],
                 "gap": model["log_loss"] - mkt["log_loss"]})
pd.DataFrame(rows).set_index("year").round(4)

,fights,model_accuracy,market_accuracy,model_log_loss,market_log_loss,gap
year,,,,,,
2019,402,0.5572,0.6331,0.6780,0.6286,0.0493
2020,347,0.6369,0.6542,0.6492,0.6167,0.0325
2021,392,0.6352,0.6492,0.6533,0.6298,0.0235
2022,397,0.6272,0.6902,0.6499,0.5949,0.0550
2019-2022,1538,0.6131,0.6567,0.6579,0.6175,0.0404


## 5. Does the model know anything the market doesn't?

The 50/50 blend above did worse than the market alone. But a blend that gives a weaker
forecaster half the say is bound to hurt, so that doesn't settle it. The fair test
**learns** the mix:

1. On 2019–2022 (out-of-fold, men's fights with odds), fit the best weights for combining
   the two, in log-odds: `score = w_market × logit(market) + w_model × logit(model)`. No
   intercept, so neither corner gets a free edge.
2. Freeze the weights and score the blend on 2023–24, against the market alone.

If `w_model` comes out near 0, or the learned blend doesn't beat the market on 2023–24,
the model adds nothing the market hasn't already priced in.

In [8]:
from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression

def features(p_market, p_model):
    clip = lambda p: np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.column_stack([logit(clip(p_market)), logit(clip(p_model))])

mix = LogisticRegression(C=np.inf, fit_intercept=False)      # C=inf: no penalty, just the best fit
mix.fit(features(c["p_market"], c["p_model"]), c["fighter_a_won"].astype(bool))
w_market, w_model = mix.coef_[0]
print(f"learned on {len(c):,} fights (2019-2022): w_market = {w_market:.3f}, w_model = {w_model:.3f}")

# the out-of-fold predictions were uncalibrated, so the frozen weights apply to the uncalibrated model
learned = expit(features(h["p_market"], h["p_model_raw"]) @ mix.coef_[0])
result = compare_log_loss(y, learned, h["p_market"])
print(f"2023-24: learned blend {score(y, learned)['log_loss']:.4f} vs market {score(y, h['p_market'])['log_loss']:.4f}  "
      f"(difference {result['difference']:+.4f}, standard error {result['se']:.4f}, p = {result['p_value']:.3f})")

learned on 1,538 fights (2019-2022): w_market = 1.090, w_model = 0.204
2023-24: learned blend 0.5802 vs market 0.5847  (difference -0.0044, standard error 0.0024, p = 0.068)


## What it shows

1. **The market is well ahead, and further than estimated.** On the 664 men's validation
   fights with odds: market 69.4% accuracy and 0.585 log loss, our model 62.4% and 0.643.
   A gap of **0.058**, five and a half standard errors: not luck. The earlier estimate
   (0.62–0.63 for the market) was right for 2014–2022, but in 2023–24 the market was
   sharper than ever (0.586 and 0.591).
2. **It's consistent, not one bad period.** Out of fold on 2019–2022 (1,538 men's fights)
   the gap is 0.040, between 0.023 and 0.055 depending on the year.
3. **Where they disagree, the market wins.** They picked different winners in 173 fights:
   the market was right in 109, the model in 64.
4. **The model covers 46% of the distance** from a coin flip to the market.
5. **The model may know a little the market underweights, but that isn't proven.** A
   50/50 blend is worse than the market alone. A blend with weights learned on 2019–2022
   gives the model a small say (0.20 against 1.09) and beats the market on 2023–24 by
   0.004, with p = 0.07: a hint, not a result.
6. **Coverage is the main caveat.** 23% of validation fights have no odds in the source.
   Those fights were harder for our model (0.675 against 0.643) and include slightly more
   debutants (22% against 19%), so the head-to-head speaks for the fights with odds.

The market knows things our data can't: injuries, weight cuts, short-notice changes, camp
news, and the money of people who follow all of it. A model built on public fight stats
was never likely to match it; the useful question now is what it would take to close part
of the gap.